# Laboratorio 4 Búsqueda en entornos complejos 
Sharis Barrios <br>
Emilio Reyes 22674

In [2]:
# librerias 
import random
import time

In [3]:
def es_solucion(estado):
    """
    Verifica si un estado representa una solución válida al problema
    de las 8 reinas.

    Parámetros:
        estado (list): lista de 8 enteros, donde el índice representa
                       la columna y el valor representa la fila.

    Retorna:
        bool: True si es una solución válida, False en caso contrario.
    """
    if not isinstance(estado, list) or len(estado) != 8:
        return False

    # Verificar que todas las filas sean enteros entre 0 y 7
    for fila in estado:
        if not isinstance(fila, int) or fila < 0 or fila > 7:
            return False

    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            # Misma fila
            if r1 == r2:
                return False

            # Misma diagonal
            if abs(r1 - r2) == abs(c1 - c2):
                return False

    return True

print(es_solucion([0, 4, 7, 5, 2, 6, 1, 3]))  # True
print(es_solucion([0, 1, 2, 3, 4, 5, 6, 7]))  # False


def heuristica(estado):
    """
    Calcula el número de pares de reinas que se atacan entre sí.
    Un estado solución tiene heurística 0.
    """
    conflictos = 0
    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            if r1 == r2 or abs(r1 - r2) == abs(c1 - c2):
                conflictos += 1

    return conflictos

True
False


In [4]:
# Generar estado aleatorio 
def estado_aleatorio():
    return [random.randint(0,7) for _ in range(8)]

In [5]:
# Generar vecinos 
def generar_vecinos(estado):

    vecinos = []

    for columna in range(8):

        fila_actual = estado[columna]

        for nueva_fila in range(8):

            if nueva_fila != fila_actual:

                nuevo_estado = estado.copy()
                nuevo_estado[columna] = nueva_fila

                vecinos.append(nuevo_estado)

    return vecinos

## Hill Climbing 

In [ ]:
def hill_climbing():
    estado = estado_aleatorio()
    pasos = 0

    while True:
        h_actual = heuristica(estado)

        # Si ya es solución
        if h_actual == 0:
            return True, pasos, h_actual

        # Generar vecinos
        vecinos = generar_vecinos(estado)

        # Elegir el mejor vecino
        mejor_vecino = min(vecinos, key=heuristica)
        h_mejor = heuristica(mejor_vecino)

        # Si no mejora, nos quedamos atrapados
        if h_mejor >= h_actual:
            return False, pasos, h_actual

        # Moverse al mejor vecino
        estado = mejor_vecino
        pasos += 1

In [9]:
for _ in range(20):
    print(hill_climbing())

#imprime si llega a la solución, pasos y heuristica final

(False, 5, 1)
(False, 3, 2)
(True, 3, 0)
(False, 2, 2)
(False, 4, 1)
(False, 3, 3)
(False, 3, 2)
(False, 4, 1)
(False, 2, 1)
(False, 5, 1)
(False, 4, 1)
(True, 2, 0)
(False, 2, 2)
(False, 3, 1)
(False, 4, 1)
(False, 3, 1)
(False, 4, 1)
(False, 3, 2)
(False, 3, 1)
(False, 2, 3)


## Hill Climbing (Random Restart)

In [10]:
def random_restart(max_restarts=100):
    total_pasos = 0

    for _ in range(max_restarts):
        exito, pasos, h = hill_climbing()
        total_pasos += pasos

        if exito:
            return True, total_pasos, 0

    # Si nunca encontró solución
    return False, total_pasos, h

In [11]:
for _ in range(20):
    print(random_restart())

(True, 11, 0)
(True, 25, 0)
(True, 4, 0)
(True, 11, 0)
(True, 32, 0)
(True, 2, 0)
(True, 7, 0)
(True, 32, 0)
(True, 43, 0)
(True, 5, 0)
(True, 9, 0)
(True, 6, 0)
(True, 5, 0)
(True, 38, 0)
(True, 9, 0)
(True, 9, 0)
(True, 4, 0)
(True, 10, 0)
(True, 4, 0)
(True, 10, 0)


## Beam Search

In [12]:
def beam_search(k=3, max_iter=100):
    # k estados iniciales
    estados = [estado_aleatorio() for _ in range(k)]
    pasos = 0

    for _ in range(max_iter):
        # Revisar si ya hay solución
        for estado in estados:
            if heuristica(estado) == 0:
                return True, pasos, 0

        # Generar todos los vecinos de todos los estados
        todos_vecinos = []
        for estado in estados:
            vecinos = generar_vecinos(estado)
            todos_vecinos.extend(vecinos)

        # Elegir los k mejores vecinos
        todos_vecinos.sort(key=heuristica)
        estados = todos_vecinos[:k]

        pasos += 1

    # Si no encontró solución
    mejor_h = min(heuristica(e) for e in estados)
    return False, pasos, mejor_h

In [14]:
for _ in range(20):
    print(beam_search(k=5))

(False, 100, 1)
(True, 4, 0)
(True, 3, 0)
(True, 4, 0)
(True, 4, 0)
(True, 3, 0)
(True, 5, 0)
(True, 5, 0)
(True, 4, 0)
(True, 3, 0)
(True, 3, 0)
(False, 100, 1)
(False, 100, 1)
(True, 3, 0)
(True, 3, 0)
(True, 7, 0)
(True, 7, 0)
(True, 4, 0)
(False, 100, 1)
(True, 3, 0)


## Experimentos

In [15]:
def experimento(algoritmo, n=1000):
    exitos = 0
    pasos_total = 0
    heuristica_total = 0

    for _ in range(n):
        exito, pasos, h = algoritmo()

        if exito:
            exitos += 1

        pasos_total += pasos
        heuristica_total += h

    return {
        "porcentaje_exito": exitos / n,
        "pasos_promedio": pasos_total / n,
        "heuristica_promedio": heuristica_total / n
    }

In [17]:
def beam_k():
    return beam_search(k=5)

In [18]:
print("Hill Climbing:")
print(experimento(hill_climbing))

print("\nRandom Restart:")
print(experimento(random_restart))

print("\nBeam Search (k=3):")
print(experimento(beam_k))

Hill Climbing:
{'porcentaje_exito': 0.164, 'pasos_promedio': 3.209, 'heuristica_promedio': 1.237}

Random Restart:
{'porcentaje_exito': 1.0, 'pasos_promedio': 22.045, 'heuristica_promedio': 0.0}

Beam Search (k=3):
{'porcentaje_exito': 0.705, 'pasos_promedio': 32.512, 'heuristica_promedio': 0.299}
